In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
characters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ', '\n', '.', ',', '“', '”', '’', '-', '?', '—', '!']

# Some Parameters
inputWindow = 50 # Model given input as 50 characters one at a time
trainSegmentLen = 400
trainWindowNum = int(trainSegmentLen/inputWindow)

testSegmentLen = 100
testWindowNum = int(testSegmentLen/inputWindow)


In [3]:
# Reading Train and Test Data and labels
with open("trainData.txt") as f:
    trainSegments = f.read().split("|")[:-1]

with open("testData.txt") as f:
    testSegments = f.read().split("|")[:-1]

labels = pd.read_csv("encryptionLabels.csv")
trainLabels = [i for i in labels["train"]]
testLabels = [i for i in labels["test"]]

print(len(trainSegments))

charToIntMap = {c:i for i,c in enumerate(characters)}


1470


In [4]:
Xtrain = []
Ytrain = []
for i in range(len(trainSegments)):
    for j in range(trainWindowNum):
        window = trainSegments[i][j * inputWindow: (j+1) * inputWindow]
        Xtrain.append([charToIntMap[c] for c in window])
        Ytrain.append(trainLabels[i])
print(len(Xtrain))
print(len(Ytrain))
print(Xtrain[0])
print(Ytrain[0])

11760
11760
[6, 31, 28, 13, 2, 4, 1, 33, 28, 26, 6, 13, 30, 7, 6, 28, 0, 25, 28, 4, 30, 13, 28, 25, 1, 1, 34, 13, 1, 29, 13, 6, 31, 28, 13, 30, 4, 28, 24, 6, 13, 30, 24, 6, 5, 25, 11, 14, 13, 13]
24


In [5]:
Xtest = []
Ytest = []
for i in range(len(testSegments)):
    for j in range(testWindowNum):
        window = testSegments[i][j * inputWindow: (j+1) * inputWindow]
        Xtest.append([charToIntMap[c] for c in window])
        Ytest.append(testLabels[i])
print(len(Xtest))
print(len(Ytest))
print(Xtest[0])
print(Ytest[0])

2940
2940
[25, 23, 20, 10, 16, 22, 18, 4, 7, 7, 22, 3, 33, 17, 0, 22, 15, 10, 22, 35, 3, 0, 35, 6, 22, 15, 3, 0, 22, 7, 33, 18, 14, 22, 10, 1, 22, 15, 3, 0, 22, 35, 10, 16, 9, 15, 13, 20, 22, 18]
33


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
Xtrain = torch.tensor(Xtrain, dtype=torch.long).to(device)
Ytrain = torch.tensor(Ytrain, dtype=torch.long).to(device)
Xtest = torch.tensor(Xtest, dtype=torch.long).to(device)
Ytest = torch.tensor(Ytest, dtype=torch.long).to(device)

In [7]:
# The MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(37, 16)

        self.fc1 = nn.Linear(inputWindow * 16, 128)
        self.fc2 = nn.Linear(128, 37)

    def forward(self, x):
        x = self.embedding(x)
        x = x.reshape(x.shape[0], -1)
        
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [8]:
#Hyper Parameters
input_size = inputWindow
output_size = 37
neurons = 128
learning_rate = 0.01

# initialize network
model = MLP().to(device)
print(model)

# Loss and Optimiser
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

MLP(
  (embedding): Embedding(37, 16)
  (fc1): Linear(in_features=800, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=37, bias=True)
)


In [9]:
# Training loop 
for i in range(1000):
    outputs = model(Xtrain)
    loss = criterion(outputs, Ytrain)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss.item():.8f}")

Iteration 0: Loss = 3.65056872
Iteration 100: Loss = 0.00011206
Iteration 200: Loss = 0.00005497
Iteration 300: Loss = 0.00003541
Iteration 400: Loss = 0.00002550
Iteration 500: Loss = 0.00001956
Iteration 600: Loss = 0.00001561
Iteration 700: Loss = 0.00001280
Iteration 800: Loss = 0.00001071
Iteration 900: Loss = 0.00000910


In [10]:
model.eval()
with torch.no_grad():
    outputs = model(Xtest)
    predictions = torch.argmax(outputs, dim=1)
    accuracy = (predictions == Ytest).float().mean()

print(f"Test Accuracy: {accuracy.item()*100:.2f}%")

Test Accuracy: 96.29%
